In [ ]:
import math
from tqdm.notebook import tqdm
import sisl
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In TBtrans and TranSiesta one is capable of performing real-space transport calculations by using real-space self-energies (see [here](https://doi.org/10.1103/PhysRevB.100.195417)).  
Currently, the real-space self-energy calculation *has* to be performed in `sisl` since it is not implemented in TranSiesta.

A real-space self-energy is a $\mathbf k$-averaged self-energy which can emulate *any* 2D or 3D electrode. I.e. for an STM junction a tip and a surface. In such a system, the surface could be modelled using the real-space self-energy to remove mirror effects of the STM tip periodic images. This is important since the distance between periodic images disturbs the calculation due to long range potential effects.

The basic principle for calculating the real-space self-energy is the Brillouin zone integral:
\begin{equation}
   \mathbf G_{\mathcal R}(E) = \int_{\mathrm{BZ}}\mathbf G_\mathbf k(E) = \int_{\mathrm{BZ}} \big[E \mathbf S_{\mathbf k} - \mathbf H_{\mathbf k}\big]^{-1}
\end{equation}
Where $\mathbf G_{\mathcal R}$ is the real-space Green function, *E* is the energy, and $\mathbf G_\mathbf k$, $\mathbf S_{\mathbf k}$ and $\mathbf H_{\mathbf k}$ are the Green function, overlap matrix and Hamiltonian, respectively, at a given **k**-point. Integration is done over the entire Brillouin-zone ($\mathrm{BZ}$). 

In this example, we will construct a STM tip probing a graphene flake.


---

This example is rather complicated and is the reason why basically everything is already done for you. Please try and understand each step.

We start by creating the graphene tight-binding model.

In [ ]:
graphene = sisl.geom.graphene(orthogonal=True)

In [ ]:
# Graphene tight-binding parameters
on, nn = 0, -2.7
H_minimal = sisl.Hamiltonian(graphene)
H_minimal.construct([[0.1, 1.43], [on, nn]])

Once the minimal graphene unit-cell (here orthogonal) is created, we now turn to the calculation of the real-space self-energy.  
The construction of this object is somewhat complicated and has a set of required input options:
- `object`: the Hamiltonian
- `semi_axis`: which axis to use for the recursive calculation of the self-energy
- `k_axes`: which axes to integrate in the Brillouin zone
- `unfold`: how many times the `object` needs to be unfolded along each lattice vector, this is an integer vector of length three

In [ ]:
# object = H_minimal
# semi_axis = 0, x-axis uses recursive self-energy calculation
# k_axes = 1, y-axis uses a Brillouin zone integral
# unfold = (10, 10, 1), the full real-space Green function is equivalent to the system
#                       H_minimal.tile(10, 0).tile(10, 1)
RSSE = sisl.RealSpaceSE(H_minimal, 0, 1, (10, 10, 1))

Now we can create the real-space self-energy.  
In TBtrans (and TranSiesta) the electrode atomic indices *must* be in consecutive order.
This is a little troublesome since the natural order in a device would be in order according to $x$, $y$ or $z$. To create the correct order we extract the *real-space coupling* matrix which is where the real-space self-energy would live. The self-energy ($\boldsymbol\Sigma^{\mathcal R}$) is calculated using:
\begin{align}
   \mathbf G^{\mathcal R}(E) &= \big[E \mathbf S^{\mathcal R} - \mathbf H^{\mathcal R} - \boldsymbol\Sigma^{\mathcal R}\big]^{-1}
   \\
   \boldsymbol\Sigma^{\mathcal R}(E) &= E \mathbf S^{\mathcal R} - \mathbf H^{\mathcal R} - \Big[\mathbf G^{\mathcal R}(E)\Big]^{-1}.
\end{align}
Where $\mathbf S^{\mathcal R}$ and $\mathbf H^{\mathcal R}$ are the real-space overlap matrix and Hamiltonian, respectively. Another way to calculate the self-energy would be to transfer the Green function from the infinite bulk into the region of interest:
\begin{equation}
   \boldsymbol\Sigma^{\mathcal R} = \mathbf V_{\infty\setminus\mathcal R,\mathcal R}\mathbf G_{\mathcal R,\infty\setminus\mathcal R}\mathbf V_{\infty\setminus\mathcal R,\mathcal R}.
\end{equation}
Where $\mathbf V_{\infty\setminus\mathcal R,\mathcal R}$ is the **to_be_filled** and $\mathbf G_{\mathcal R,\infty\setminus\mathcal R}$ is the **to_be_filled**. From the second equation it is obvious that the self-energy only lives on the boundary that $\mathbf V_{\infty\setminus\mathcal R,\mathcal R}$ couples to. Exactly this region is extracted using `real_space_coupling` as below. Take some time to draw a simple 2D lattice coupling and confirm the area that the real-space self-energies couple to.
*HINT:* Draw the unit-cell for a 4x4 square lattice with nearest-neighbour couplings, then mark the atoms that couple to atoms outside the unit-cell.  

----

In this example, we also retrieve the indices for the electrode atoms, those that connect *out to the infinite plane*.

In [ ]:
H_elec, elec_indices = RSSE.real_space_coupling(ret_indices=True)
# By default a real-space Hamiltonian holds no couplings, however, tbtrans is not
# aware that this is *intended* for this operation, so we have to force some supercells
# so tbtrans can use it. It's a hack... Nothing more.
H_elec.set_nsc((3, 3, 1))
H_elec.write("GRAPHENE.nc")

The above yields the electrode region which contains the self-energies. The full device region is nothing but the `H_minimal` tiled $10\times10$ times with an attached STM tip on top. Here we need to arrange the electrode atoms first, and then the final atoms of the device region after. The `real_space_parent` method returns the Hamiltonian that obeys the *unfolded* size - in this case $10\times10$ times larger. One should always use this method to get the correct device order of atoms, since the order of tiling is determined by the `semi_axes` and `k_axis` arguments.

In [ ]:
H = RSSE.real_space_parent()
# Create the true device by re-arranging the atoms
indices = np.arange(len(H))
indices = np.delete(indices, elec_indices)
# first electrodes, then rest of device
indices = np.concatenate([elec_indices, indices])
# Now re-arange matrix
H = H.sub(indices)

Lastly, we need to add the STM tip. Here we simply add a gold atom and manually add the hoppings. Since this is tight-binding, we have full control over the self-energy and potential landscape. Therefore we don't need to extend the electrode region to screen-off the tip region. In DFT systems a screening region is required.

In [ ]:
STM = sisl.Geometry(
    [0, 0, 0],
    atoms=sisl.Atom("Au", R=1.0001),
    lattice=sisl.Lattice([10, 10, 1], nsc=[1, 1, 3]),
)
H_STM = sisl.Hamiltonian(STM)
H_STM.construct([(0.1, 1.1), (0, -0.75)])
H_STM.write("STM.nc")

In [ ]:
mid_xyz = H.geometry.center()
# The center atom in the device that we place the STM tip above
idx = H.close(mid_xyz, R=1.33)[0]
# The place of the STM tip (once merged the index of the first atom of the STM)
idx_STM = len(H)
H_device = H.add(H_STM, offset=H.geometry.xyz[idx] - H_STM.geometry.xyz[0] + [0, 0, 2])
H_device[idx_STM, idx] = -0.1
H_device[idx, idx_STM] = -0.1

In [ ]:
H_device.write("DEVICE.nc")

Before we can run calculations, we need to create the real-space self-energy for the graphene flake in `sisl`.
Since the algorithm is not implemented in TBtrans (nor TranSiesta) it needs to be done here.

Also, lets visualize how the system actually looks:

In [ ]:
# Let's visualize where we put the self-energies
H_device.geometry.plot(
    axes="xy",
    show_cell=False,
    atoms_scale=0.7,
    atoms_style={"color": "cyan", "atoms": range(len(H_elec))},
)

This is somewhat complicated since the files require a specific order. For ease, this tutorial implements it for you.

In [ ]:
# A real-space transport calculation ONLY needs the Gamma-point
gamma = sisl.MonkhorstPack(H_elec, [1, 1, 1])
# Energy-contour
dE = 0.04
E = np.arange(-2, 2 + dE / 2, dE)
sisl.io.tableSile("contour.E", "w").write_data(E, np.zeros(E.size) + dE)

In [ ]:
# Now create the file (should take around 3-4 minutes)
eta = 0.001 * 1j
with sisl.io.tbtgfSileTBtrans("GRAPHENE.TBTGF") as f:
    f.write_header(gamma, E + eta)
    for ispin, new_k, k, e in tqdm(f, unit="rsSE"):
        if new_k:
            f.write_hamiltonian(H_elec.Hk(format="array", dtype=np.complex128))
        SeHSE = RSSE.self_energy(e + eta, bulk=True, coupling=True)
        f.write_self_energy(SeHSE)

## A pure `sisl` implementation of the transport algorithm

Instead of doing transport calculations in TBtrans, we can test the internal algorithms in `sisl` that can effectively do the same.
These tools can readily read an fdf file and create the necessary NEGF device for us.
It resides as a toolbox for sisl (`sisl_toolbox`) but is intended to be merged into `sisl` once the API has stabilized and made a bit more generic.

Let us try to do it:

In [ ]:
from sisl_toolbox.btd import (
    DeviceGreen,
    BlochElectrode,
    DownfoldElectrode,
    RealSpaceElectrode,
)
from sisl.physics import RecursiveSI

# First we should check the interface for instantiating the class
# 1. The device Hamiltonian
# 2. a list of electrodes
#    Each item of the electrode consists of:
#    (electrode object, atomic indices in the device)
elecs = []
# The first electrode is the STM
STM_elec = BlochElectrode(RecursiveSI(H_STM, "+C"), name="STM")
STM_indices = range(len(H), len(H_device))
elecs.append([STM_elec, STM_indices])

# Now comes the RealSpaceElectrode which is a bit more tricky
# First define it, note we re-use the RSSE object from above!
RSSE_elec = RealSpaceElectrode(RSSE, name="graphene")
RSSE_indices = range(len(H_elec))
# The indices of this electrode are the first atoms corresponding to H_elec
elecs.append([RSSE_elec, RSSE_indices])

G = DeviceGreen(H_device, elecs)
T = G.transmission(0.5, "STM", "graphene")
print(f"transmission E=0.5  = {T}")

There's a lot going on here. But this is to show you that one can essentially bypass TBtrans completely. Since `sisl` is much simpler to extend, one can do lots of explorative calculations. E.g. it supports natively spin-orbit.

Since you have been exposed to the BTD algorithm, let's try and visualize how the algorithm does this for you.
This code will extract the blocks, and show the different blocks on a different colouring scale.
The BTD algorithm will invert each differently colored block separately, it only ever holds the matrix elements that exists between two neighboring colored atoms, which yields its performance.

In [ ]:
def plot_btd(device, cmap="gist_ncar"):
    import matplotlib

    cmap = matplotlib.colormaps[cmap]

    # Now extract the plots
    permute = device.permute
    blocks = device.blocks
    nblocks = len(blocks)
    geom = G.H.geometry

    atoms_style = []
    off = 0
    for i, block in enumerate(blocks):
        atoms = permute[off : off + block]
        style = {
            "atoms": atoms,
            "color": matplotlib.colors.to_hex(cmap(i / nblocks)),
        }
        atoms_style.append(style)
        off += block

    return geom.plot(
        atoms_style=atoms_style, axes="xy", atoms_scale=0.7, show_cell=False
    )


plot_btd(G)

## Exercises

- Calculate transport, density-of-states and bond-transmissions.  
   Please search the manual on how to edit the `RUN.fdf` according to the following:
   - Force `tbtrans` to use the generated `TBTGF` file. This is the same as in [TB_07](../TB_07/TB_07.ipynb) example, i.e. using the `out-of-core` tag
   - Force `tbtrans` to use an energy grid defined in an external file (`contour.E`). Search the `tbtrans` manual.
   
- Plot the bond-transmissions and check their symmetry, does the symmetry depend on the injection point?
- Is there a particular reason for choosing `semi_axis` and `k_axes` as they are chosen? Or could they be swapped?

- **TIME**: Redo the calculations using three electrodes (left/right/tip) using *k*-points. Converge transmission and then plot the bond-transmissions.  
   Do they look as the real-space calculation? If they are the same, why?

In [ ]:
!tbtrans RUNfix.fdf > TBT.out

In [ ]:
tbt = sisl.get_sile("siesta.TBT.nc")

## Learned lessons

- Creation of real-space self-energies
- Manual specification of energy points for TBtrans
- Manual creation of self-energy files for TBtrans
- Forcing TBtrans to use an existing **TBTGF** file on disk, previously created with `sisl`.